# Convolutional Neural Network

### Importing the libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
tf.__version__

'2.21.0'

In [3]:
# Ubah parameter di sini untuk eksperimen
# DIUBAH: ukuran gambar diturunkan dari 150x150 ke 128x128 agar training di VS Code/CPU lebih cepat.
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
# DIUBAH LANJUTAN: epoch dinaikkan dari 10 ke 20 agar akurasi punya kesempatan naik, tetapi tetap jauh lebih cepat dari 60 epoch.
EPOCHS = 20
# DIUBAH: filter awal diturunkan dari 64 ke 32; konsep filter CNN tetap sesuai instruksi tugas.
FILTERS = 32        # Coba variasi laporan: 16, 32, 64
KERNEL_SIZE = 3     # Coba variasi laporan: 3 atau 5
L2_REG = 0.000005   # Dikurangi drastis — model tidak overfitting, regularisasi sebelumnya terlalu kuat
PRED_IMAGE_PATHS = ['single_prediction/cat_or_dog_1.jpg', 'single_prediction/cat_or_dog_2.jpg']
RANDOM_SEED = 42
tf.keras.utils.set_random_seed(RANDOM_SEED)

## Part 1 - Data Preprocessing

### Preprocessing the Training set

In [4]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    # DIUBAH: augmentasi inti dari instruksi dosen tetap ada, tetapi dibuat lebih ringan.
    shear_range=0.1,        # dimiringkan
    zoom_range=0.15,        # zoom
    horizontal_flip=True,   # flip
    rotation_range=180,     # diputar sampai 180 derajat sesuai instruksi
    fill_mode='nearest'
)
training_set = train_datagen.flow_from_directory('training_set',
                                                 target_size=IMAGE_SIZE,
                                                 batch_size=BATCH_SIZE,
                                                 class_mode='binary')

Found 8000 images belonging to 2 classes.


### Preprocessing the Test set

In [5]:
test_datagen = ImageDataGenerator(rescale = 1./255)
test_set = test_datagen.flow_from_directory('test_set',
                                            target_size = IMAGE_SIZE,
                                            batch_size = BATCH_SIZE,
                                            class_mode = 'binary')

Found 2000 images belonging to 2 classes.


### Initialising the CNN

In [6]:
cnn = tf.keras.models.Sequential()

### Step 1 - Convolution

In [7]:
# Block 1: double conv (VGG-style) — 2 conv sebelum pooling, fitur lebih kaya
cnn.add(tf.keras.layers.Conv2D(
    filters=FILTERS, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG),
    input_shape=[IMAGE_SIZE[0], IMAGE_SIZE[1], 3]
))
cnn.add(tf.keras.layers.Conv2D(
    filters=FILTERS, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)
))
cnn.add(tf.keras.layers.BatchNormalization())

c:\Users\gitoe\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### Step 2 - Pooling

In [8]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Adding a second convolutional layer

In [9]:
# Block 2: FILTERS*2, double conv
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 2, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 2, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.BatchNormalization())
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

# Block 3: FILTERS*4, double conv
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 4, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 4, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.BatchNormalization())
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

# Block 4: FILTERS*4, double conv
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 4, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 4, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.BatchNormalization())
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Step 3 - Flattening

In [10]:
cnn.add(tf.keras.layers.Flatten())

### Step 4 - Full Connection

In [11]:
cnn.add(tf.keras.layers.Dense(units=256, activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Dropout(0.4))
cnn.add(tf.keras.layers.Dense(units=128, activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Dropout(0.2))

### Step 5 - Output Layer

In [12]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

## Part 3 - Training the CNN

### Compiling the CNN

In [13]:
# DIUBAH LANJUTAN 2: learning rate Adam diperkecil dari default 0.001 ke 0.0005 agar training lebih stabil.
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)
cnn.compile(optimizer = optimizer, loss = 'binary_crossentropy', metrics = ['accuracy'])

In [14]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# DIUBAH LANJUTAN 2: monitor diarahkan ke val_accuracy karena target laporan adalah akurasi.
# Patience dinaikkan agar model tidak berhenti terlalu cepat setelah learning rate diturunkan.
early_stop = EarlyStopping(monitor='val_accuracy', mode='max', patience=8, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_accuracy', mode='max', factor=0.5, patience=3, min_lr=1e-6, verbose=1)

history = cnn.fit(x=training_set, validation_data=test_set, epochs=EPOCHS, callbacks=[early_stop, reduce_lr])

best_epoch = history.history['val_accuracy'].index(max(history.history['val_accuracy'])) + 1
train_acc_pct = history.history['accuracy'][best_epoch - 1] * 100
val_acc_pct = max(history.history['val_accuracy']) * 100
print(f'Best epoch: {best_epoch}')
print(f'Best train acc: {train_acc_pct:.2f}%')
print(f'Best val acc: {val_acc_pct:.2f}%')

Epoch 1/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 55s 209ms/step - accuracy: 0.5640 - loss: 0.8158 - val_accuracy: 0.5000 - val_loss: 0.8494 - learning_rate: 5.0000e-04
Epoch 2/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 51s 205ms/step - accuracy: 0.5865 - loss: 0.6977 - val_accuracy: 0.5315 - val_loss: 0.8791 - learning_rate: 5.0000e-04
Epoch 3/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 51s 204ms/step - accuracy: 0.6119 - loss: 0.6678 - val_accuracy: 0.5880 - val_loss: 0.7301 - learning_rate: 5.0000e-04
Epoch 4/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 51s 204ms/step - accuracy: 0.6217 - loss: 0.6580 - val_accuracy: 0.6315 - val_loss: 0.6439 - learning_rate: 5.0000e-04
Epoch 5/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 51s 202ms/step - accuracy: 0.6699 - loss: 0.6255 - val_accuracy: 0.6735 - val_loss: 0.6332 - learning_rate: 5.0000e-04
Epoch 6/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 52s 209ms/step - accuracy: 0.6867 - loss: 0.6025 - val_accuracy: 0.6375 - val_loss: 0.7195 - learning_rate: 5.0000e-04
Epoch 7/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 49s 19

## Part 4 - Making a single prediction

In [15]:
import numpy as np
from tensorflow.keras.preprocessing import image

class_indices = training_set.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}

print('class_indices:', class_indices)
for pred_path in PRED_IMAGE_PATHS:
    test_image = image.load_img(pred_path, target_size = IMAGE_SIZE)
    test_image = image.img_to_array(test_image)
    test_image = np.expand_dims(test_image, axis = 0)
    test_image = test_image / 255.0

    result = cnn.predict(test_image, verbose = 0)
    probability = float(result[0][0])
    predicted_index = 1 if probability >= 0.5 else 0
    predicted_label = idx_to_class[predicted_index]

    file_name = pred_path.split('/')[-1]
    print(f"{file_name}: prob={probability:.4f}, pred={predicted_label}")

class_indices: {'cats': 0, 'dogs': 1}
cat_or_dog_1.jpg: prob=0.9629, pred=dogs
cat_or_dog_2.jpg: prob=0.0964, pred=cats


In [16]:
# Dinonaktifkan: output prediksi sekarang sudah dicetak per file di cell sebelumnya
